In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers torch  tqdm



import pandas as pd
import numpy as np
from transformers import pipeline
import torch
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

In [4]:
with open('/content/drive/MyDrive/Movie-Reviews-Analysis/data/test_data.pkl', 'rb') as f:
    test_data = pickle.load(f)

X_test = test_data['X_test']
y_test = test_data['y_test']
best_model_acc = test_data['best_acc']

print(f"Test set: {len(X_test):,} reviews")
print(f"Traditional model accuracy: {best_model_acc:.4f}")

Test set: 10,000 reviews
Traditional model accuracy: 0.9008


In [5]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1,
)

candidate_labels = ["positive", "negative"]
print(f"Labels: {candidate_labels}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Labels: ['positive', 'negative']


In [6]:
sample_size = 50
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test.iloc[sample_idx].values
y_sample = y_test[sample_idx]

print(f"Class distribution in sample:")
unique, counts = np.unique(y_sample, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {'Positive' if u==1 else 'Negative'}: {c}")

Class distribution in sample:
  Negative: 20
  Positive: 30


In [7]:
predictions = []
confidences = []
times = []


for i, text in enumerate(X_sample):
    start = time.time()

    result = classifier(text, candidate_labels)

    pred = 1 if result['labels'][0] == 'positive' else 0
    conf = result['scores'][0]

    predictions.append(pred)
    confidences.append(conf)
    times.append(time.time() - start)

    if (i+1) % 20 == 0:
        print(f"  Processed {i+1}/{sample_size}")

print(f"\nDone! Avg time: {np.mean(times)*1000:.1f}ms per review")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Processed 20/50
  Processed 40/50

Done! Avg time: 156.9ms per review


In [8]:
print("\n" + "="*100)
print("SAMPLE PREDICTIONS")
print("="*100)

# Show 10 random examples
show_idx = np.random.choice(sample_size, 10, replace=False)

for i, idx in enumerate(show_idx):
    text = X_sample[idx][:150] + "..." if len(X_sample[idx]) > 150 else X_sample[idx]
    true = "Positive" if y_sample[idx] == 1 else "Negative"
    pred = "Positive" if predictions[idx] == 1 else "Negative"
    conf = confidences[idx]

    print(f"\n{i+1}. {text}")
    print(f"   True: {true} | Pred: {pred} (conf: {conf:.3f}) {'✓' if y_sample[idx]==predictions[idx] else '✗'}")


SAMPLE PREDICTIONS

1. "Father of the Pride " was another of those good shows that unfortunately don't have a very long life . And that is pretty sad ,specially if you consi...
   True: Positive | Pred: Negative (conf: 0.597) ✗

2. What a dreadful movie. The effects were poor, especially by todays standards, but that was forgivable. What was unforgivable was the terrible rehashin...
   True: Negative | Pred: Negative (conf: 0.934) ✓

3. This is the latest entry in the long series of films with the French agent, O.S.S. 117 (the French answer to James Bond). The series was launched in t...
   True: Positive | Pred: Positive (conf: 0.551) ✓

4. I don't think any movie of Van Damme's will ever beat Universal Soldier but u never know. This movie was good but not as good as 1st. VD returns a Luc...
   True: Negative | Pred: Positive (conf: 0.561) ✗

5. Lazy writing, bad acting and wooden direction lead us to a 2005 Canadian-made TV movie called HUSH, not to be confused with about eight othe

In [9]:
mis_idx = np.where(np.array(predictions) != y_sample)[0]

print(f"\nMisclassified: {len(mis_idx)}/{sample_size} ({len(mis_idx)/sample_size:.1%})")

if len(mis_idx) > 0:
    print("\nFirst 3 misclassified examples:")
    for i, idx in enumerate(mis_idx[:3]):
        text = X_sample[idx][:150] + "..." if len(X_sample[idx]) > 150 else X_sample[idx]
        true = "Positive" if y_sample[idx] == 1 else "Negative"
        pred = "Positive" if predictions[idx] == 1 else "Negative"
        conf = confidences[idx]

        print(f"\n{i+1}. {text}")
        print(f"   True: {true} | Pred: {pred} (conf: {conf:.3f})")


Misclassified: 7/50 (14.0%)

First 3 misclassified examples:

1. I don't know what the rest of you guys watch Steven Seagal movies for, but I watch them because, as silly as they are, they're at least always good fo...
   True: Negative | Pred: Positive (conf: 0.532)

2. "Father of the Pride " was another of those good shows that unfortunately don't have a very long life . And that is pretty sad ,specially if you consi...
   True: Positive | Pred: Negative (conf: 0.597)

3. Clearly this film was made for a newer generation that may or may not have had an inkling of Charles Bukowski's work. The autobiographical Henry China...
   True: Negative | Pred: Positive (conf: 0.643)


In [11]:
accuracy = np.mean(np.array(predictions) == y_sample)

print("\n" + "="*60)
print("ZERO-SHOT CLASSIFICATION SUMMARY")
print("="*60)
print(f"""
Model: facebook/bart-large-mnli
Sample size: {sample_size} reviews
Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)
Traditional model: {best_model_acc:.4f} ({best_model_acc*100:.2f}%)
Average confidence: {np.mean(confidences):.4f}
""")


ZERO-SHOT CLASSIFICATION SUMMARY

Model: facebook/bart-large-mnli
Sample size: 50 reviews
Accuracy: 0.8600 (86.00%)
Traditional model: 0.9008 (90.08%)
Average confidence: 0.8292

